# **Programa Especializado en Credit Scoring con Python**
<img src="../../figuras/logo.png" width="200"/>

## 📊 **Sesión 8: Implementación del Score y Uso en la Toma de Decisiones.**

**Docente**: Enzo Infantes Zúñiga  
**Contacto**: <enzo.infantes28@gmail.com>  
**LinkedIn**: [enzo-infantes](https://www.linkedin.com/in/enzo-infantes/)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sys
import os
import warnings
import joblib
from fastapi import FastAPI
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
warnings.filterwarnings("ignore")

np.random.seed(42)

absolute_path = os.path.dirname(os.path.dirname(os.getcwd()))
data_path = os.path.join(absolute_path, "data", "s08")
src_path_07 = os.path.join(absolute_path, "src", "s07")
src_path_08 = os.path.join(absolute_path, "src", "s08")
model_path = os.path.join(absolute_path, "models", "s08")
figure_path = os.path.join(absolute_path, "figuras", "s08")
sys.path.insert(0, src_path_07)
sys.path.insert(0, src_path_08)

from datetime import datetime
from preprocessing import Preprocessing
from feature_engineering import FeatureEngineering
from modeling import Modeling

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 🎯 **Simulación de Decisión Crediticia**

- Implementación Simulada con FastAPI

# **1. Simulación de Decisión Crediticia**

```
JSON input
   ↓
preprocessing.py
   ↓
feature_engineering.py
   ↓
modelo.pkl
   ↓
predict_proba
   ↓
scorecard
   ↓
cutoff
   ↓
decision
```

In [ ]:
app = FastAPI()

model = joblib.load(os.path.join(model_path, 'logistic_regression_v2.pkl'))

# parámetros del score
PDO = 20
Score0 = 600
Odds0 = (1 - 0.29) / 0.29

factor = PDO / np.log(2)
offset = Score0 - factor * np.log(Odds0)

cutoff = 566.7437

@app.post("/predict")

def predict(data: list):

    df = pd.DataFrame(data)

    df = Preprocessing(df)

    df = FeatureEngineering(df)

    pds = model.predict_proba(df)[:,1]

    odds = (1 - pds) / pds

    scores = offset + factor * np.log(odds)

    decisions = np.where(
        scores >= cutoff,
        "Approve",
        "Reject"
    )

    response = []

    for i in range(len(df)):

        response.append({
            "pd": float(pds[i]),
            "score": float(scores[i]),
            "decision": decisions[i]
        })

    return response